# 11. SQL CTEs & Recursive Hierarchies: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **11. SQL CTEs & Recursive Hierarchies**. Complex data pipelines require modular, readable query topologies. Common Table Expressions (`WITH` clause) allow defining named temporary result sets that exist only within query execution. This notebook covers single and multi-CTE chained pipelines, recursive CTEs (`WITH RECURSIVE`) for traversing hierarchical graphs, generating continuous synthetic date sequences, and recursive cycle detection.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Standard Non-Recursive CTEs: `WITH cte_name AS (...)`
- [x] 🔹 Multi-Stage Chained Pipelines: Composing Sequential CTEs
- [x] 🔹 Recursive CTE Architecture: `WITH RECURSIVE` Anchor and Recursive Members
- [x] 🔹 Synthetic Sequence Generation (Date Series & Gap Fillers)
- [x] 🔍 Scenario: Multi-Tier Merchant Settlement Fee Reconciliation Pipeline








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Standard Non-Recursive CTEs: `WITH cte_name AS (...)`
- **What it does:** Creates a named virtual relation within the scope of a single query statement.
- **Syntax:** `WITH cte_name AS (SELECT cols FROM table WHERE condition) SELECT * FROM cte_name;`
- **Dataset Application & Code Demonstration:** Pre-filters high-value transactions before calculating customer statistics.


In [2]:
%%sql
WITH high_value_tx AS (
    SELECT customer_id, transaction_amount, region
    FROM transactions
    WHERE transaction_amount > 1500.00
)
SELECT 
    region,
    COUNT(*) AS high_value_count,
    ROUND(SUM(transaction_amount), 2) AS total_high_value_volume
FROM high_value_tx
GROUP BY region
ORDER BY total_high_value_volume DESC;


,region,high_value_count,total_high_value_volume
0,West,871,1531311.58
1,East,863,1510923.24
2,North,851,1494175.87
3,South,832,1461511.85
4,South,26,46142.21
5,west,24,42662.70
6,south,25,42544.01
7,east,22,39500.43
8,north,18,32173.98
9,East,19,32069.20


### 🔹 Multi-Stage Chained Pipelines: Composing Sequential CTEs
- **What it does:** Declares multiple comma-separated CTEs where downstream CTEs can directly query upstream CTEs.
- **Syntax:** `WITH cte_1 AS (...), cte_2 AS (SELECT * FROM cte_1 ...) SELECT * FROM cte_2;`
- **Dataset Application & Code Demonstration:** Builds a 3-step customer spend aggregation pipeline.


In [3]:
%%sql
WITH customer_totals AS (
    SELECT 
        customer_id, 
        COUNT(transaction_id) AS tx_count,
        SUM(transaction_amount) AS total_spent
    FROM transactions
    GROUP BY customer_id
),
customer_segmented AS (
    SELECT 
        customer_id,
        tx_count,
        total_spent,
        CASE 
            WHEN total_spent > 15000.00 THEN 'PLATINUM'
            WHEN total_spent > 8000.00 THEN 'GOLD'
            ELSE 'STANDARD'
        END AS loyalty_tier
    FROM customer_totals
)
SELECT 
    loyalty_tier,
    COUNT(customer_id) AS customer_count,
    ROUND(AVG(total_spent), 2) AS avg_tier_spend
FROM customer_segmented
GROUP BY loyalty_tier;


,loyalty_tier,customer_count,avg_tier_spend
0,GOLD,864,11011.60
1,PLATINUM,102,16836.08
2,STANDARD,523,5918.99


### 🔹 Recursive CTE Architecture: `WITH RECURSIVE`
- **What it does:** Executes an anchor query followed by a recursive step that repeatedly executes until the termination condition is reached.
- **Syntax:** `WITH RECURSIVE cte AS (SELECT anchor UNION ALL SELECT recursive FROM cte WHERE termination_cond) SELECT * FROM cte;`
- **Dataset Application & Code Demonstration:** Generates a synthetic 7-day calendar date sequence.


In [4]:
%%sql
WITH RECURSIVE calendar_dates(day_date, step_num) AS (
    SELECT DATE('2024-01-01'), 1
    UNION ALL
    SELECT DATE(day_date, '+1 day'), step_num + 1
    FROM calendar_dates
    WHERE step_num < 7
)
SELECT day_date, step_num FROM calendar_dates;


,day_date,step_num
0,2024-01-01,1
1,2024-01-02,2
2,2024-01-03,3
3,2024-01-04,4
4,2024-01-05,5
5,2024-01-06,6
6,2024-01-07,7


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Hierarchical Organizational Chart Traversal with Recursive CTEs
- **Objective:** Model a manager-employee hierarchy and calculate reporting depth levels.
- **Approach:** Traverse parent-child relationships recursively from top executive to individual contributors.


In [5]:
%%sql
CREATE TABLE IF NOT EXISTS org_hierarchy (
    emp_id TEXT PRIMARY KEY,
    name TEXT,
    manager_id TEXT
);
INSERT OR REPLACE INTO org_hierarchy VALUES
    ('E001', 'CEO Sarah', NULL),
    ('E002', 'VP Alex', 'E001'),
    ('E003', 'VP Dave', 'E001'),
    ('E004', 'Lead Eng Emma', 'E002');

WITH RECURSIVE org_tree AS (
    SELECT emp_id, name, manager_id, 0 AS depth_level, name AS reporting_path
    FROM org_hierarchy
    WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, e.manager_id, t.depth_level + 1, t.reporting_path || ' -> ' || e.name
    FROM org_hierarchy e
    INNER JOIN org_tree t ON e.manager_id = t.emp_id
)
SELECT emp_id, name, depth_level, reporting_path
FROM org_tree
ORDER BY depth_level ASC;


'Query Executed Successfully.'